# Notebook 02 — Data Cleaning
## Machine Learning-based Late Delivery Risk Prediction in Global Supply Chain Operations
**Client:** APL Logistics (KWE Group) | **Platform:** Unified Mentor

### Objective
This notebook covers data cleaning — dropping leakage columns, removing PII and irrelevant columns, handling missing values, fixing data types, and saving the cleaned dataset for use in all subsequent notebooks.

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

In [2]:
# Load Data

data = pd.read_csv('../data/apl_logistics.csv', encoding='latin-1')
df = data.copy()
print(f"Raw Dataset Shape: {df.shape}")

Raw Dataset Shape: (180519, 40)


## Drop Leakage Columns
These columns are post-event information — they would not be available at the time of prediction. Using them would cause data leakage and result in artificially inflated model performance.

In [3]:
leakage_cols = ['Delivery Status', 'Order Status']

df = df.drop(columns=leakage_cols)
print(f"Dropped leakage columns: {leakage_cols}")
print(f"Shape after dropping leakage columns: {df.shape}")

Dropped leakage columns: ['Delivery Status', 'Order Status']
Shape after dropping leakage columns: (180519, 38)


##  Check for Duplicates

In [4]:
duplicates_before = df.duplicated().sum()
print(f"Duplicate rows found: {duplicates_before}")

if duplicates_before > 0:
    df = df.drop_duplicates()
    print(f"Duplicates removed. New shape: {df.shape}")
else:
    print("No duplicate rows found. No action needed.")

Duplicate rows found: 0
No duplicate rows found. No action needed.


##  Handle Missing Values
From Notebook 01, only two columns had missing values — both are already dropped. Verifying no missing values remain.

In [5]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['Missing Count'] > 0]

if missing.empty:
    print("No missing values remaining in the dataset.")
else:
    print(f"Columns still with missing values:")
    display(missing)

Columns still with missing values:


,Missing Count,Missing %
Customer Lname,8,0.0
Customer Zipcode,3,0.0


##  Drop PII and Irrelevant Columns
These columns are either personally identifiable information (PII), ID columns with no predictive value, or columns with near-zero correlation with the target variable confirmed in Notebook 01.

In [6]:
irrelevant_cols = [
    'Customer Fname',       # PII, no modeling value
    'Customer Lname',       # PII, no modeling value
    'Customer Street',      # PII, high cardinality noise
    'Customer Zipcode',     # PII, near-zero correlation
    'Latitude',             # Near-zero correlation, redundant with region
    'Longitude',            # Near-zero correlation, redundant with region
    'Category Id',          # ID column
    'Department Id',        # ID column
    'Customer Id',          # ID column
    'Order Customer Id',    # ID column
]

df = df.drop(columns=irrelevant_cols)
print(f"Dropped irrelevant columns: {irrelevant_cols}")
print(f"Shape after dropping irrelevant columns: {df.shape}")

Dropped irrelevant columns: ['Customer Fname', 'Customer Lname', 'Customer Street', 'Customer Zipcode', 'Latitude', 'Longitude', 'Category Id', 'Department Id', 'Customer Id', 'Order Customer Id']
Shape after dropping irrelevant columns: (180519, 28)


##  Fix Data Types
Verifying all columns have correct data types. Categorical columns should be object/string, numerical columns should be int or float.

In [7]:
print("Current Data Types:")
print("=" * 50)
print(df.dtypes)

Current Data Types:
Type                                 str
Days for shipping (real)           int64
Days for shipment (scheduled)      int64
Benefit per order                float64
Sales per customer               float64
Late_delivery_risk                 int64
Category Name                        str
Customer City                        str
Customer Country                     str
Customer Segment                     str
Customer State                       str
Department Name                      str
Market                               str
Order City                           str
Order Country                        str
Order Item Discount              float64
Order Item Discount Rate         float64
Order Item Product Price         float64
Order Item Profit Ratio          float64
Order Item Quantity                int64
Sales                            float64
Order Item Total                 float64
Order Profit Per Order           float64
Order Region                         

##  Final Dataset Overview

In [8]:
print(f"Final Cleaned Dataset Shape: {df.shape}")
print(f"\nRemaining Columns ({len(df.columns)}):")
for col in df.columns:
    print(f"  - {col}")

Final Cleaned Dataset Shape: (180519, 28)

Remaining Columns (28):
  - Type
  - Days for shipping (real)
  - Days for shipment (scheduled)
  - Benefit per order
  - Sales per customer
  - Late_delivery_risk
  - Category Name
  - Customer City
  - Customer Country
  - Customer Segment
  - Customer State
  - Department Name
  - Market
  - Order City
  - Order Country
  - Order Item Discount
  - Order Item Discount Rate
  - Order Item Product Price
  - Order Item Profit Ratio
  - Order Item Quantity
  - Sales
  - Order Item Total
  - Order Profit Per Order
  - Order Region
  - Order State
  - Product Name
  - Product Price
  - Shipping Mode


In [9]:
# Final missing value confirmation

print("Final Missing Value Check:")
print("=" * 50)
total_missing = df.isnull().sum().sum()
print(f"Total missing values in cleaned dataset: {total_missing}")

Final Missing Value Check:
Total missing values in cleaned dataset: 0


In [13]:
# Final check for duplicates

df.duplicated().sum()

np.int64(2)

In [14]:
df.drop_duplicates(inplace = True)

## Save Cleaned Dataset
Saving the cleaned dataset to the data/ folder. All subsequent notebooks will load this file instead of the raw data.

In [15]:
df.to_csv('../data/cleaned_data.csv', index=False)
print("Cleaned dataset saved to: data/cleaned_data.csv")
print(f"Final Shape: {df.shape}")
df.head()

Cleaned dataset saved to: data/cleaned_data.csv
Final Shape: (180517, 28)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Late_delivery_risk,Category Name,Customer City,Customer Country,Customer Segment,Customer State,Department Name,Market,Order City,Order Country,Order Item Discount,Order Item Discount Rate,Order Item Product Price,Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Order Profit Per Order,Order Region,Order State,Product Name,Product Price,Shipping Mode
0,DEBIT,6,4,159.69,472.45,1,Cardio Equipment,Brownsville,EE. UU.,Consumer,TX,Footwear,Pacific Asia,Mumbai,India,27.50,0.06,99.99,0.34,5,499.95,472.45,159.69,South Asia,Maharashtra,Nike Men's Free 5.0+ Running Shoe,99.99,Standard Class
1,DEBIT,4,4,48.71,167.96,0,Shop By Sport,Littleton,EE. UU.,Consumer,CO,Golf,LATAM,San Pedro Sula,Honduras,31.99,0.16,39.99,0.29,5,199.95,167.96,48.71,Central America,Cortés,Under Armour Girls' Toddler Spine Surge Runni,39.99,Standard Class
2,DEBIT,4,4,87.36,181.99,0,Water Sports,Littleton,EE. UU.,Consumer,CO,Fan Shop,LATAM,San Pedro Sula,Honduras,18.00,0.09,199.99,0.48,1,199.99,181.99,87.36,Central America,Cortés,Pelican Sunstream 100 Kayak,199.99,Standard Class
3,DEBIT,6,4,-41.89,175.99,1,Water Sports,Littleton,EE. UU.,Consumer,CO,Fan Shop,USCA,New York City,Estados Unidos,24.00,0.12,199.99,-0.24,1,199.99,175.99,-41.89,East of USA,Nueva York,Pelican Sunstream 100 Kayak,199.99,Standard Class
4,DEBIT,6,4,10.00,40.00,1,Women's Apparel,Littleton,EE. UU.,Consumer,CO,Golf,USCA,New York City,Estados Unidos,10.00,0.20,50.00,0.25,1,50.00,40.00,10.00,East of USA,Nueva York,Nike Men's Dri-FIT Victory Golf Polo,50.00,Standard Class
